# MapReduce in Python 

This notebook provides a hands-on introduction to the **MapReduce** paradigm using Python.  
We build the three core functions **from scratch** — `map_func`, `shuffle_and_sort`, and `reduce_func` — and then apply them to real movie data.

### Dataset
We will use the same dataset from last week

### Part 1 — Load the Dataset

In [ ]:
import pandas as pd
import string

# Load the dataset — adjust the relative path if needed
df = pd.read_csv("../week01/wiki_movie_plots_deduped.csv")

# Quick sanity check
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(3)

### Part 2 — The MapReduce Framework

Before writing any task-specific logic, we build three **reusable** functions that form the backbone of every MapReduce job we will run today.

| Function | Role | Input | Output |
|---|---|---|---|
| `map_func` | User-defined; emits key-value pairs | A single record | `[(key, value), ...]` |
| `shuffle_and_sort` | Groups all values by key | `[(key, value), ...]` | `{key: [v1, v2, ...]}` |
| `reduce_func` | User-defined; aggregates values per key | `{key: [values]}` | `{key: result}` |

> **Why is Shuffle & Sort a separate step?**  
> In a real distributed system this is the *most expensive* network operation — every mapper on every machine ships its emitted pairs to the correct reducer node. We have one machine, but the logic is identical.

In [ ]:
from collections import defaultdict

# ── 1. Shuffle & Sort ────────────────────────────────────────────────────────
def shuffle_and_sort(mapped_output):
    """
    Groups all emitted (key, value) pairs produced by the Map phase by their key.

    This is the crucial intermediate step between Map and Reduce.
    In a real distributed system, this involves:
      - Sorting all emitted pairs across all worker nodes
      - Shipping each key to the single reducer responsible for it
    Here we mimic that by grouping into a dictionary of lists.

    Args:
        mapped_output: a flat list of (key, value) tuples from the map phase.

    Returns:
        A dict  {key: [value1, value2, ...]}
    """
    grouped = defaultdict(list)
    for key, value in mapped_output:
        grouped[key].append(value)
    return dict(grouped)


# ── 2. Generic run_mapreduce orchestrator ────────────────────────────────────
def run_mapreduce(records, map_func, reduce_func):
    """
    Runs a full MapReduce job.

    Args:
        records   : iterable of input records (e.g. rows from a DataFrame).
        map_func  : callable(record) -> [(key, value), ...].
        reduce_func: callable(key, values) -> (key, result).

    Returns:
        List of (key, result) tuples.
    """
    # ── Map Phase ────────────────────────────────────────────────────────────
    mapped = []
    for record in records:
        mapped.extend(map_func(record))         # each record may emit many pairs

    # ── Shuffle & Sort Phase ─────────────────────────────────────────────────
    shuffled = shuffle_and_sort(mapped)

    # ── Reduce Phase ─────────────────────────────────────────────────────────
    results = []
    for key, values in shuffled.items():
        results.append(reduce_func(key, values))

    return results


print("Core MapReduce functions defined ✓")

## Part 3 — Task 1: Word Count on Movie Titles

**Goal:** Count how many times each word appears across all movie titles in the dataset.

This is the "Hello, World!" of MapReduce.

### The plan

```
INPUT ROW        →  MAP                     →  SHUFFLE & SORT          →  REDUCE
─────────────────────────────────────────────────────────────────────────────────
"The Dark Knight"   ("the", 1)               "the":    [1, 1, 1, ...]    ("the",    N)
                    ("dark", 1)              "dark":   [1, 1, ...]        ("dark",   N)
                    ("knight", 1)            "knight": [1, ...]           ("knight", N)
```

**Preprocessing** in the mapper:
- Strip leading/trailing whitespace
- Convert to lowercase
- Remove punctuation

In [ ]:
# ── Mapper ───────────────────────────────────────────────────────────────────
def title_word_mapper(title):
    """
    Takes a single movie title (string) and emits one (word, 1) pair per word.

    Preprocessing steps:
      1. Strip whitespace
      2. Convert to lowercase (so 'The' and 'the' are treated as the same word)
      3. Remove punctuation (e.g. colons, hyphens that remain after splitting)
    """
    # Remove punctuation using str.translate — fast and readable
    # TODO
    # Split on whitespace and emit a (word, 1) pair for every non-empty token


# ── Reducer ───────────────────────────────────────────────────────────────────
def word_count_reducer(key, values):
    """
    Receives a word (key) and a list of 1s (one per occurrence).
    Returns the (word, total_count) pair.
    """
    # TODO

# ── Run ───────────────────────────────────────────────────────────────────────
# The 'records' we pass to run_mapreduce are the individual title strings
titles = df["Title"].dropna().tolist()

results_task1 = None # TODO

# Sort by count descending to show the most common words first
results_task1_sorted = list(None) # TODO

print("Top 20 most frequent words in movie titles:\n")
for word, count in results_task1_sorted[:20]:
    print(f"  {word:<20} {count}")


The top results from Task 1 are dominated by **function words** like *"the"*, *"a"*, *"of"* — words that carry little meaning. These are called **stop words**.

We will now use the [NLTK](https://www.nltk.org/) library's built-in English stop-word list to filter them out in the mapper before emitting any pairs. All that changes is the mapper — the shuffle/sort and reducer stay exactly the same. This is a key strength of the MapReduce design: **each phase is independent and swappable**.

> **Why filter in the mapper?**  
> In a real cluster, filtering early (in the mapper) means less data is shipped over the network during the Shuffle phase — which is the most expensive step.

In [ ]:
import nltk

# Download the stopwords corpus (only needs to run once per machine)
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords

# Load into a Python set — O(1) lookup is important when filtering millions of words
STOP_WORDS = set(stopwords.words("english"))

print(f"Stop words loaded: {len(STOP_WORDS)} words")
print("Sample:", sorted(STOP_WORDS)[:10])

In [ ]:
# ── Mapper (with stop-word filtering) ────────────────────────────────────────
def title_word_mapper_no_stopwords(title):
    """
    Same as title_word_mapper, but also filters out NLTK English stop words.

    Only the mapper changes — shuffle_and_sort and word_count_reducer are reused
    exactly as defined in Part 2. This demonstrates the modularity of MapReduce.
    """
    # TODO


# ── Run ───────────────────────────────────────────────────────────────────────
# reducer is unchanged — we reuse word_count_reducer from Task 1
results_task1b = None # TODO

results_task1b_sorted = list(None) # TODO

print("Top 20 most frequent MEANINGFUL words in movie titles:\n")
for word, count in results_task1b_sorted[:20]:
    print(f"  {word:<20} {count}")

# ── Comparison ────────────────────────────────────────────────────────────────
print(f"\nUnique words before stop-word removal : {len(results_task1_sorted):,}")
print(f"Unique words after  stop-word removal : {len(results_task1b_sorted):,}")
print(f"Words filtered out                    : {len(results_task1_sorted) - len(results_task1b_sorted):,}")

## Task 2 — Movies per Decade

Concept: Modifying the key before emitting.
Map: Read the Release Year, calculate the decade (e.g., 1901 -> 1900s), and emit (Decade, 1).
Reduce: Sum the values to get the total number of movies per decade.

In [ ]:
# ── Mapper ───────────────────────────────────────────────────────────────────
def decade_mapper(row):
    """
    Takes a DataFrame row (as a named tuple / dict-like) and emits (decade_label, 1).

    The key transformation:  release_year  →  "1900s", "1910s", etc.
    We floor to the nearest 10 using integer division: 1901 // 10 * 10 = 1900
    """
    # try:
    #     # TODO something
    # except # TODO something
    # return # TODO


# ── Reducer ───────────────────────────────────────────────────────────────────
def count_reducer(key, values):
    """Generic count reducer — sums a list of 1s."""
    return None # TODO


# ── Run ───────────────────────────────────────────────────────────────────────
# Pass rows as dicts so the mapper can access columns by name
rows = df.to_dict(orient="records")

results_task2 = list(None) # TODO

# Sort chronologically
results_task2_sorted = sorted(results_task2, key=lambda x: x[0])

print("Number of movies per decade:\n")
for decade, count in results_task2_sorted:
    print(f"  {decade:<10} {count}")


## Task 3 — Genre → Movie Title Inverted Index

- **Concept**: Aggregating data into a list rather than a mathematical sum.
- **Map**: Read Genre and Title. Emit (Genre, Title).
- **Reduce**: Instead of adding numbers, append the titles into a list. Emit (Genre, [Title1, Title2, ...]).

In [ ]:
# ── Mapper ───────────────────────────────────────────────────────────────────
def genre_title_mapper(row):
    """
    Emits (genre, title) — one pair per row.
    Normalises the genre label to lowercase and strips whitespace.
    """
    # TODO


# ── Reducer ───────────────────────────────────────────────────────────────────
def list_collect_reducer(key, values):
    """
    Instead of summing, we keep all values as a sorted list.
    This is fundamentally different from all previous reducers.
    """
    # TODO


# ── Run ───────────────────────────────────────────────────────────────────────
results_task3 = list(None) # TODO

# Sort by genre name for readability
results_task3_sorted = sorted(results_task3, key=lambda x: x[0])

print("Inverted index — Genre → Movie Titles (showing first 3 titles per genre):\n")
for genre, titles in results_task3_sorted[100:110]:
    preview = titles[:3]
    print(f"  {genre:<20}  ({len(titles):,} movies)  e.g. {preview}")

## Now its your turn!


### Task 1 

- Calculate the Average Plot Length per Origin/Ethnicity

**Concept:** The mapper emits a *tuple* as the value — not just a `1`.  
Without this trick, the reducer cannot compute an average because it would have lost how many movies contributed to a total.

**The naïve wrong approach:**  
Emit `(Origin, word_count)` → reduce by summing → this gives the *total* words, not the *average*.

**The correct approach — emit a (sum, count) pair:**
```
MAP emits:     (Origin, (word_count, 1))
SHUFFLE gives: Origin → [(500, 1), (320, 1), (410, 1), ...]
REDUCE does:   total_words = sum of all first elements
               total_movies = sum of all second elements
               average = total_words / total_movies
```

> This pattern — emitting partial aggregates — is fundamental to distributed computing.  
> You will see it again in frameworks like Spark and Flink.

In [ ]:
# ── Mapper ───────────────────────────────────────────────────────────────────
def plot_length_mapper(row):
    """
    Emits (origin, (word_count, 1)) — a tuple as the value.

    word_count counts the whitespace-separated tokens in the plot.
    The '1' is the movie count contribution from this single row.
    Both are needed by the reducer to compute the average.
    """
    origin     = None # TODO
    plot       = None # TODO
    word_count = None # TODO
    return None # TODO


# ── Reducer ───────────────────────────────────────────────────────────────────
def avg_plot_length_reducer(key, values):
    """
    values is a list of (word_count, 1) tuples — one per movie for this origin.

    We unzip them into two lists, then compute the average:
        average = sum(word_counts) / sum(movie_counts)
    """
    total_words  = None # TODO
    total_movies = None # TODO
    average      = None # TODO
    return None # TODO


# ── Run ───────────────────────────────────────────────────────────────────────
results_task4 = None # TODO

# Sort by average plot length descending
results_task4_sorted = sorted(results_task4, key=lambda x: x[1], reverse=True)

print("Average plot length (words) per origin — sorted descending:\n")
for origin, avg in results_task4_sorted:
    print(f"  {origin:<30}  {avg:>8.1f} words")


### Task 2 

- Calculate Most Prolific Director per Decade (2 MapReduce Jobs)

**Concept:** Some problems cannot be solved in a single MapReduce pass.  
Here we need two chained jobs:

```
JOB 1: Count movies per (Decade, Director)
─────────────────────────────────────────────────────────────────────────
MAP 1    →  ((decade, director), 1)
REDUCE 1 →  ((decade, director), total_movies)

JOB 2: Find the top director for each decade
─────────────────────────────────────────────────────────────────────────
MAP 2    →  (decade, (director, total_movies))   ← reshape the key
REDUCE 2 →  (decade, top_director)               ← argmax over values
```

**Why can't we do this in one pass?**  
After Job 1 the data looks like `{("1940s", "John Ford"): 12, ("1940s", "Howard Hawks"): 8, ...}`.  
To find the *maximum within a decade* we need to group by decade alone — which requires a second shuffle.

> **Note on the `Director` column:** It sometimes contains multiple comma-separated names  
> (e.g. `"George S. Fleming, Edwin S. Porter"`). The mapper splits on commas and credits  
> each director individually.

#### Job 1 count movies per (decade, director)

In [ ]:
def director_decade_mapper(row):
    """
    Emits ((decade_label, director_name), 1) for every director credited on a film.

    A single movie can credit multiple directors (comma-separated), so this mapper
    may emit more than one pair per input row.
    """
    try:
       # something
    except # something
    raw_directors = None # TODO
    # Split on comma; strip whitespace; skip empty/placeholder values
    directors = None # TODO

    if not directors:
        directors = ["Unknown"]

    return None # TODO


def sum_reducer(key, values):
    """Generic summation reducer — works for any numeric list."""
    # TODO


job1_results = None # TODO
# job1_results is a list of  ((decade, director), count)



#### JOB 2 — find the top director within each decade

In [ ]:

def top_director_mapper(record):
    """
    Reshapes the output of Job 1.
    Input:  ((decade, director), total_movies)
    Output: (decade, (director, total_movies))

    We change the key from (decade, director) → decade only so that the
    shuffle phase groups ALL directors for a decade together.
    """
    (decade, director), total_movies = None # TODO
    return None # TODO


def top_director_reducer(key, values):
    """
    Receives all (director, total_movies) tuples for a single decade.
    Returns the director with the highest movie count (argmax).
    """
    best_director, best_count = None # TODO
    return None # TODO


job2_results = None # TODO

# Sort chronologically
job2_sorted = list(None) # TODO

print("Most prolific director per decade:\n")
for decade, info in job2_sorted:
    print(f"  {decade}  →  {info}")

### Task 3
- Calculate Directors with the Most Diverse Genre Portfolio (2 MapReduce Jobs)

**Concept:** The classic **Count Distinct** problem in distributed systems.

**Why can't this be done in one pass?**  
If we emit `(director, 1)` once per movie, a director with 5 westerns would be counted 5 times for that genre.  
We need to deduplicate `(Director, Genre)` pairs *before* counting.

```
JOB 1: Deduplicate (Director, Genre) pairs
─────────────────────────────────────────────────────────────────────────────────
MAP 1    →  ((director, genre), 1)       ← composite key
REDUCE 1 →  (director, 1)               ← ONE '1' per unique pair, regardless of sum

JOB 2: Count how many unique genres each director has worked in
─────────────────────────────────────────────────────────────────────────────────
MAP 2    →  (director, 1)               ← identity mapper
REDUCE 2 →  (director, total_unique_genres)
```

**The trick in Job 1:** The reducer ignores the sum entirely.  
The *grouping by composite key* in the shuffle phase is what performs the deduplication — each unique `(director, genre)` pair arrives at exactly one reducer, which emits exactly one `1`.

#### Job 1: Deduplicate (Director, Genre) pairs

In [ ]:

def director_genre_mapper(row):
    """
    Emit a ((director, genre), 1) pair for each director involved in a movie.
    - Splits multi-director cells on commas (e.g. "Spielberg, Lucas").
    - Uses a composite key so the shuffle groups by (director, genre) — this
      is what deduplicates: a director who made 10 westerns still yields only
      ONE entry for ("Director", "western") after the shuffle.
    """
    directors = None # TODO
    genre = None # TODO
    return None # TODO


def dedup_reducer(key, values):
    """
    key    = (director, genre)  — already unique after shuffle
    values = [1, 1, ...]        — how many movies; we IGNORE the count
    Returns (director, 1) — one credit toward genre diversity, guaranteed unique.
    """
    director, genre = None # TODO
    return None # TODO


job1_t6 = None # TODO

print(f"Job 1 produced {len(job1_t6):,} unique (director, genre) pairs")
print("Sample:", job1_t6[:5])



#### Job 2: Sum unique-genre credits per director

In [ ]:


def identity_mapper_t6(record):
    """
    The input is already a (director, 1) tuple from Job 1.
    Nothing to transform — just wrap it in a list as our framework expects.
    """
    director, count = None # TODO
    return None # TODO


def sum_reducer_t6(key, values):
    """
    key    = director name
    values = [1, 1, 1, ...]  — one '1' per unique genre from Job 1
    Returns (director, total_unique_genres).
    """
    return None # TODO


job2_t6 = None # TODO

# ── Results ─────────────────────────────────────────────────────────────────
top_diverse = sorted(job2_t6, key=lambda x: x[1], reverse=True)

print("\n🎬 Top 20 Directors by Genre Diversity")
print(f"{'Director':<30} {'Unique Genres':>13}")
print("-" * 45)
for director, genres in top_diverse[:20]:
    print(f"{director:<30} {genres:>13}")